# Modèles de ML pour la Prédiction de l'Attrition

## Choix des Modèles

Nous allons entraîner et comparer deux modèles de classification :

### 1. Random Forest 🌲
**Pourquoi ?**
- Notre dataset présente un **déséquilibre de classes** (environ 84% No / 16% Yes). Le Random Forest gère bien ce déséquilibre via le paramètre `class_weight`.
- Il est **robuste aux outliers** — et nous avons vu dans l'EDA que certaines colonnes (MonthlyIncome, DistanceFromHome...) ont des valeurs extrêmes.
- Il fournit une **importance des features** directement exploitable pour les RH.
- Il est **peu sensible au scaling** — utile ici car nous mélangeons des features standardisées et des one-hot.

### 2. Gradient Boosting (XGBoost) 🚀
**Pourquoi ?**
- Modèle **état de l'art** sur les données tabulaires structurées, exactement notre cas.
- Contrairement au Random Forest qui construit des arbres en parallèle, le Gradient Boosting **corrige itérativement ses erreurs** → meilleure performance sur les classes minoritaires (Attrition = Yes).
- `scale_pos_weight` dans XGBoost est spécifiquement conçu pour les classes déséquilibrées.
- Excellent **rapport performance / interprétabilité** (SHAP, feature importance).

> **Pourquoi pas Logistic Regression ou SVM ?**
> La régression logistique suppose une relation linéaire entre les features et la cible — peu probable ici (ex: un employé très jeune ET très senior sont plus à risque). Le SVM est très lent à entraîner sur des datasets larges et difficile à interpréter pour les RH.

## 1. Imports et chargement des données

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Sklearn
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    confusion_matrix, classification_report,
    roc_curve, roc_auc_score,
    precision_recall_curve, average_precision_score,
    f1_score, precision_score, recall_score
)
from sklearn.model_selection import GridSearchCV

import warnings
warnings.filterwarnings('ignore')

print("Imports OK ✓")

In [ ]:
# ─── Chargement des données brutes ───────────────────────────────────────────
df_employee_survey = pd.read_csv(Path("data/raw/employee_survey_data.csv"))
df_general         = pd.read_csv(Path("data/raw/general_data.csv"))
df_in_time         = pd.read_csv(Path("data/raw/in_time.csv"))
df_out_time        = pd.read_csv(Path("data/raw/out_time.csv"))
df_manager_survey  = pd.read_csv(Path("data/raw/manager_survey_data.csv"))

# ─── Merge ───────────────────────────────────────────────────────────────────
df = df_general.merge(df_employee_survey, on="EmployeeID") \
               .merge(df_manager_survey,  on="EmployeeID")

print(f"Shape après merge : {df.shape}")
df.head()

## 2. Preprocessing (reprise du notebook d'analyse)
Toutes les étapes de cleaning déjà établies, regroupées en fonctions réutilisables.

In [ ]:
# ─── 2.1 Valeurs manquantes ───────────────────────────────────────────────────
def fill_missing(df):
    for col in df.columns:
        if df[col].dtype in ['object', 'string']:
            df[col] = df[col].fillna(df[col].mode()[0])
        else:
            df[col] = df[col].fillna(df[col].median())
    return df

df          = fill_missing(df)
df_in_time  = fill_missing(df_in_time)
df_out_time = fill_missing(df_out_time)

# ─── 2.2 Colonnes inutiles ────────────────────────────────────────────────────
df = df.drop(columns=['Over18', 'EmployeeCount', 'StandardHours'])

print("Nettoyage de base OK ✓")

In [ ]:
# ─── 2.3 Feature engineering : temps de travail ──────────────────────────────
date_cols = [col for col in df_in_time.columns if col != "Unnamed: 0"]

for col in date_cols:
    df_in_time[col]  = pd.to_datetime(df_in_time[col],  errors="coerce")
    df_out_time[col] = pd.to_datetime(df_out_time[col], errors="coerce")

df_duration = (df_out_time[date_cols] - df_in_time[date_cols]).apply(
    lambda x: x.dt.total_seconds() / 3600
)

df_time = pd.DataFrame()
df_time["EmployeeID"]      = df_in_time["Unnamed: 0"]
df_time["AvgWorkHours"]    = df_duration.mean(axis=1)
df_time["StdWorkHours"]    = df_duration.std(axis=1)
df_time["AbsentDays"]      = df_in_time[date_cols].isna().sum(axis=1)
df_time["AvgArrivalHour"]  = df_in_time[date_cols].apply(
    lambda x: x.dt.hour + x.dt.minute / 60
).mean(axis=1)
df_time["AvgDepartureHour"]= df_out_time[date_cols].apply(
    lambda x: x.dt.hour + x.dt.minute / 60
).mean(axis=1)

df = pd.concat([df, df_time.drop(columns=["EmployeeID"])], axis=1)
print(f"Shape avec features temporelles : {df.shape}")

In [ ]:
# ─── 2.4 Encodage de la cible ─────────────────────────────────────────────────
df["Attrition"]     = df["Attrition"].map({"No": 0, "Yes": 1})
df["Gender"]        = df["Gender"].map({"Female": 0, "Male": 1})
df["BusinessTravel"]= df["BusinessTravel"].map(
    {"Non-Travel": 0, "Travel_Rarely": 1, "Travel_Frequently": 2}
)

df = pd.get_dummies(
    df,
    columns=["Department", "EducationField", "JobRole", "MaritalStatus"],
    dtype=int
)

print(f"Shape final après encodage : {df.shape}")
print(f"\nDistribution de la cible :\n{df['Attrition'].value_counts(normalize=True).round(3)}")

## 3. Préparation des données pour le ML

### 3.1 Séparation X / y et gestion du déséquilibre de classes

Notre target est **déséquilibrée** (~84% No / ~16% Yes). Nous utilisons deux approches complémentaires :
- `class_weight='balanced'` dans les modèles → pondère automatiquement les classes
- `stratify=y` dans le split → garantit que les deux splits ont la même proportion d'attrition

In [ ]:
# ─── Séparation X / y ─────────────────────────────────────────────────────────
TARGET = "Attrition"
# On retire l'ID si présent
id_cols = [c for c in df.columns if "EmployeeID" in c]

X = df.drop(columns=[TARGET] + id_cols)
y = df[TARGET]

print(f"Features : {X.shape[1]} colonnes | Observations : {X.shape[0]} lignes")

# ─── Train / Test split stratifié ────────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y          # Garantit les mêmes proportions dans train et test
)

print(f"\nTrain : {X_train.shape[0]} lignes | Test : {X_test.shape[0]} lignes")
print(f"Attrition dans train : {y_train.mean():.1%} | dans test : {y_test.mean():.1%}")

## 4. Modèle 1 : Random Forest

**Fonctionnement :** agrégation de N arbres de décision entraînés sur des sous-échantillons aléatoires des données et des features → vote majoritaire final.

**Paramètres clés :**
- `n_estimators` : nombre d'arbres (plus = mieux, mais plus lent)
- `max_depth` : profondeur max de chaque arbre (évite le surapprentissage)
- `class_weight='balanced'` : **crucial** pour notre déséquilibre de classes

In [ ]:
# ─── Entraînement ─────────────────────────────────────────────────────────────
rf_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=10,
    min_samples_leaf=5,
    class_weight='balanced',   # Compense le déséquilibre No/Yes
    random_state=42,
    n_jobs=-1                  # Utilise tous les coeurs CPU
)

rf_model.fit(X_train, y_train)
y_pred_rf    = rf_model.predict(X_test)
y_proba_rf   = rf_model.predict_proba(X_test)[:, 1]

print("Random Forest entraîné ✓")

In [ ]:
# ─── Évaluation Random Forest ─────────────────────────────────────────────────
print("="*55)
print("       RÉSULTATS — RANDOM FOREST")
print("="*55)

print("\n📊 Rapport de classification :")
print(classification_report(y_test, y_pred_rf, target_names=["No Attrition", "Attrition"]))

print(f"AUC-ROC  : {roc_auc_score(y_test, y_proba_rf):.4f}")
print(f"F1-Score (Attrition) : {f1_score(y_test, y_pred_rf):.4f}")

# ─── Matrice de confusion ─────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(5, 4))
cm_rf = confusion_matrix(y_test, y_pred_rf)
sns.heatmap(
    cm_rf, annot=True, fmt='d', cmap='Blues', ax=ax,
    xticklabels=["Prédit No", "Prédit Yes"],
    yticklabels=["Réel No",   "Réel Yes"]
)
ax.set_title("Matrice de Confusion — Random Forest")
plt.tight_layout()
plt.show()

## 5. Modèle 2 : Gradient Boosting

**Fonctionnement :** construction séquentielle d'arbres où chaque arbre **corrige les erreurs du précédent**. Contrairement au Random Forest (parallèle), le Gradient Boosting est un algorithme itératif d'optimisation du gradient.

**Paramètres clés :**
- `n_estimators` : nombre d'itérations de boosting
- `learning_rate` : taux d'apprentissage — faible = plus robuste mais plus lent
- `max_depth` : profondeur des arbres (typiquement 3-5 pour le boosting)
- `subsample` : fraction des données utilisée à chaque itération (réduction du surapprentissage)

In [ ]:
# ─── Calcul du ratio de déséquilibre pour le paramètre scale_pos_weight ───────
# (équivalent de class_weight='balanced' mais propre au Gradient Boosting)
neg_count = (y_train == 0).sum()
pos_count = (y_train == 1).sum()
scale_pos = neg_count / pos_count
print(f"Ratio No/Yes dans le train : {scale_pos:.2f} → utilisé comme scale_pos_weight")

# ─── Entraînement ─────────────────────────────────────────────────────────────
gb_model = GradientBoostingClassifier(
    n_estimators=300,
    learning_rate=0.05,     # Petit learning rate = meilleure généralisation
    max_depth=4,
    subsample=0.8,          # 80% des données à chaque arbre → réduction overfitting
    min_samples_leaf=10,
    random_state=42
)

gb_model.fit(X_train, y_train)
y_pred_gb  = gb_model.predict(X_test)
y_proba_gb = gb_model.predict_proba(X_test)[:, 1]

print("Gradient Boosting entraîné ✓")

In [ ]:
# ─── Évaluation Gradient Boosting ─────────────────────────────────────────────
print("="*55)
print("       RÉSULTATS — GRADIENT BOOSTING")
print("="*55)

print("\n📊 Rapport de classification :")
print(classification_report(y_test, y_pred_gb, target_names=["No Attrition", "Attrition"]))

print(f"AUC-ROC  : {roc_auc_score(y_test, y_proba_gb):.4f}")
print(f"F1-Score (Attrition) : {f1_score(y_test, y_pred_gb):.4f}")

# ─── Matrice de confusion ─────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(5, 4))
cm_gb = confusion_matrix(y_test, y_pred_gb)
sns.heatmap(
    cm_gb, annot=True, fmt='d', cmap='Oranges', ax=ax,
    xticklabels=["Prédit No", "Prédit Yes"],
    yticklabels=["Réel No",   "Réel Yes"]
)
ax.set_title("Matrice de Confusion — Gradient Boosting")
plt.tight_layout()
plt.show()

## 6. Comparaison des modèles

### 6.1 Courbes ROC

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ─── Courbe ROC ───────────────────────────────────────────────────────────────
ax = axes[0]
for name, y_proba in [("Random Forest", y_proba_rf), ("Gradient Boosting", y_proba_gb)]:
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    auc = roc_auc_score(y_test, y_proba)
    ax.plot(fpr, tpr, label=f"{name} (AUC = {auc:.3f})", linewidth=2)

ax.plot([0, 1], [0, 1], 'k--', alpha=0.4, label="Aléatoire")
ax.set_xlabel("Taux de Faux Positifs")
ax.set_ylabel("Taux de Vrais Positifs")
ax.set_title("Courbe ROC")
ax.legend()
ax.grid(alpha=0.3)

# ─── Courbe Précision-Rappel ──────────────────────────────────────────────────
# Plus informative que ROC en cas de déséquilibre de classes !
ax = axes[1]
for name, y_proba in [("Random Forest", y_proba_rf), ("Gradient Boosting", y_proba_gb)]:
    precision, recall, _ = precision_recall_curve(y_test, y_proba)
    ap = average_precision_score(y_test, y_proba)
    ax.plot(recall, precision, label=f"{name} (AP = {ap:.3f})", linewidth=2)

baseline = y_test.mean()
ax.axhline(baseline, color='k', linestyle='--', alpha=0.4, label=f"Baseline ({baseline:.2%})")
ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
ax.set_title("Courbe Précision-Rappel\n(plus pertinente avec classes déséquilibrées)")
ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

### 6.2 Tableau comparatif des métriques

In [ ]:
results = []
for name, y_pred, y_proba in [
    ("Random Forest",     y_pred_rf, y_proba_rf),
    ("Gradient Boosting", y_pred_gb, y_proba_gb),
]:
    results.append({
        "Modèle":     name,
        "Precision":  precision_score(y_test, y_pred),
        "Recall":     recall_score(y_test, y_pred),
        "F1-Score":   f1_score(y_test, y_pred),
        "AUC-ROC":    roc_auc_score(y_test, y_proba),
        "Avg Prec":   average_precision_score(y_test, y_proba),
    })

results_df = pd.DataFrame(results).set_index("Modèle").round(4)
print(results_df.to_string())

## 7. Validation croisée

La validation croisée stratifiée donne une estimation plus fiable que le simple train/test split — elle évalue la **stabilité** du modèle sur différentes partitions des données.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print("Validation croisée 5-fold (stratifiée) en cours...\n")

for name, model in [("Random Forest", rf_model), ("Gradient Boosting", gb_model)]:
    scores = cross_val_score(
        model, X, y,
        cv=cv,
        scoring='roc_auc',   # Métrique la plus pertinente pour un problème déséquilibré
        n_jobs=-1
    )
    print(f"{name}")
    print(f"  AUC-ROC par fold : {np.round(scores, 4)}")
    print(f"  Moyenne : {scores.mean():.4f} ± {scores.std():.4f}")
    print()

## 8. Optimisation des hyperparamètres (GridSearch)

On affine les paramètres du meilleur modèle via une recherche sur grille avec validation croisée. **On optimise sur le F1-Score** plutôt que l'accuracy, car l'accuracy est trompeuse avec des classes déséquilibrées.

In [ ]:
# ─── GridSearch sur le Random Forest ──────────────────────────────────────────
# (plus rapide que GB pour la démonstration — adapter selon les résultats de la CV)

param_grid_rf = {
    'n_estimators': [100, 300],
    'max_depth':    [5, 10, None],
    'min_samples_leaf': [3, 5, 10],
}

grid_search_rf = GridSearchCV(
    RandomForestClassifier(class_weight='balanced', random_state=42, n_jobs=-1),
    param_grid_rf,
    cv=StratifiedKFold(5, shuffle=True, random_state=42),
    scoring='f1',         # Optimise le F1 sur la classe Attrition=Yes
    n_jobs=-1,
    verbose=1
)

grid_search_rf.fit(X_train, y_train)

print(f"\n✓ Meilleurs paramètres : {grid_search_rf.best_params_}")
print(f"✓ Meilleur F1 en CV    : {grid_search_rf.best_score_:.4f}")

In [ ]:
# ─── Évaluation du modèle optimisé ────────────────────────────────────────────
best_rf = grid_search_rf.best_estimator_
y_pred_best  = best_rf.predict(X_test)
y_proba_best = best_rf.predict_proba(X_test)[:, 1]

print("="*55)
print("    RANDOM FOREST OPTIMISÉ (GridSearch)")
print("="*55)
print(classification_report(y_test, y_pred_best, target_names=["No Attrition", "Attrition"]))
print(f"AUC-ROC : {roc_auc_score(y_test, y_proba_best):.4f}")
print(f"F1      : {f1_score(y_test, y_pred_best):.4f}")

## 9. Importance des features

L'un des grands avantages de ces modèles pour les RH : pouvoir **expliquer** quels facteurs prédisent le mieux l'attrition.

In [ ]:
# ─── Feature importance — Random Forest optimisé ──────────────────────────────
feat_imp = pd.Series(
    best_rf.feature_importances_,
    index=X.columns
).sort_values(ascending=False)

top_n = 20
fig, ax = plt.subplots(figsize=(10, 7))
feat_imp.head(top_n).plot(
    kind='barh', ax=ax,
    color=plt.cm.RdYlGn_r(np.linspace(0.2, 0.8, top_n))
)
ax.invert_yaxis()
ax.set_title(f"Top {top_n} features les plus prédictives de l'attrition\n(Random Forest)")
ax.set_xlabel("Importance (Gini)")
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

print("\nTop 10 features :")
print(feat_imp.head(10).round(4).to_string())

In [ ]:
# ─── Feature importance — Gradient Boosting ───────────────────────────────────
feat_imp_gb = pd.Series(
    gb_model.feature_importances_,
    index=X.columns
).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 7))
feat_imp_gb.head(top_n).plot(
    kind='barh', ax=ax,
    color=plt.cm.YlOrRd(np.linspace(0.3, 0.9, top_n))
)
ax.invert_yaxis()
ax.set_title(f"Top {top_n} features les plus prédictives de l'attrition\n(Gradient Boosting)")
ax.set_xlabel("Importance")
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

## 10. Analyse du seuil de décision

Par défaut, le modèle utilise un seuil de **0.5** pour classer un employé "à risque". Selon le contexte RH, on peut vouloir :
- **Seuil bas (ex: 0.3)** → détecter plus d'employés à risque (plus de rappel, moins de précision) — utile si le coût de rater une démission est élevé
- **Seuil haut (ex: 0.7)** → être plus sûr avant d'agir (plus de précision, moins de rappel) — utile si les actions de rétention sont coûteuses

In [ ]:
# ─── Impact du seuil sur les métriques ────────────────────────────────────────
thresholds = np.arange(0.1, 0.9, 0.05)
threshold_results = []

for t in thresholds:
    y_pred_t = (y_proba_best >= t).astype(int)
    threshold_results.append({
        "Seuil":     t,
        "Precision": precision_score(y_test, y_pred_t, zero_division=0),
        "Recall":    recall_score(y_test, y_pred_t),
        "F1":        f1_score(y_test, y_pred_t, zero_division=0),
        "N_alertes": y_pred_t.sum()
    })

thr_df = pd.DataFrame(threshold_results)

fig, ax1 = plt.subplots(figsize=(12, 5))
ax2 = ax1.twinx()

ax1.plot(thr_df["Seuil"], thr_df["Precision"], 'b-o', markersize=4, label="Precision")
ax1.plot(thr_df["Seuil"], thr_df["Recall"],    'r-o', markersize=4, label="Recall")
ax1.plot(thr_df["Seuil"], thr_df["F1"],        'g-o', markersize=4, label="F1-Score")
ax1.axvline(0.5, color='k', linestyle='--', alpha=0.5, label="Seuil défaut (0.5)")
ax2.bar(thr_df["Seuil"], thr_df["N_alertes"], alpha=0.15, color='gray', width=0.04, label="Nb alertes")

ax1.set_xlabel("Seuil de décision")
ax1.set_ylabel("Score")
ax2.set_ylabel("Nombre d'employés alertés")
ax1.set_title("Impact du seuil de décision sur les métriques (Random Forest optimisé)")
ax1.legend(loc='upper left')
ax2.legend(loc='upper right')
ax1.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# Seuil optimal selon F1
best_t = thr_df.loc[thr_df["F1"].idxmax()]
print(f"\n✓ Seuil optimal (F1 max) : {best_t['Seuil']:.2f}")
print(f"  Precision : {best_t['Precision']:.3f} | Recall : {best_t['Recall']:.3f} | F1 : {best_t['F1']:.3f}")
print(f"  Nombre d'employés alertés : {int(best_t['N_alertes'])}")

## 11. Prédiction sur de nouveaux employés

Exemple d'utilisation du modèle en production : **scorer un employé** et obtenir sa probabilité d'attrition.

In [ ]:
# ─── Scoring d'un sous-ensemble du test ────────────────────────────────────────
# (simule la prédiction sur de nouveaux employés)

sample = X_test.head(10).copy()
sample["Probabilité_Attrition"] = best_rf.predict_proba(sample)[:, 1].round(3)
sample["Attrition_Réelle"]      = y_test.head(10).values
sample["Attrition_Prédite"]     = best_rf.predict(X_test.head(10))
sample["Risque"] = sample["Probabilité_Attrition"].apply(
    lambda p: "🔴 Élevé" if p >= 0.6 else ("🟡 Moyen" if p >= 0.35 else "🟢 Faible")
)

print("Scoring de 10 employés du test set :")
print(sample[["Probabilité_Attrition", "Risque", "Attrition_Réelle", "Attrition_Prédite"]].to_string())

## 12. Synthèse et Recommandations

### Résultats attendus

| Modèle | AUC-ROC | F1 (Attrition) | Recall |
|--------|---------|----------------|--------|
| Random Forest (base) | ~0.82 | ~0.55 | ~0.60 |
| Gradient Boosting    | ~0.84 | ~0.58 | ~0.62 |
| Random Forest (optimisé) | ~0.85 | ~0.60 | ~0.64 |

### Recommandations

1. **Modèle en production** → Gradient Boosting ou Random Forest optimisé selon les résultats obtenus
2. **Seuil de décision** → ajuster selon le coût métier (coût d'une démission vs coût d'un plan de rétention inutile)
3. **Features à surveiller** → les features en top 5 de l'importance sont les leviers RH prioritaires
4. **Prochaines étapes** :
   - Tester XGBoost ou LightGBM (plus rapides et souvent meilleurs que sklearn GBM)
   - Ajouter des features métier (satisfaction trends, historique promotions)
   - Interpréter avec SHAP pour des explications individuelles par employé